# HW2 — Financial Data Processing

2026 cohort · [Official assignment](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework2.md) · [Submission](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw02)

**Unexecuted reproducibility notebook**. Uses the homework's explicit IPO source **iposcoop.com** (not the lecture's IPO provider) and the official Q4 Google Drive parquet. Sources change; run and verify outputs and form choices. Expected row counts are approximations, not assertions.

In [ ]:
%pip -q install pandas numpy requests lxml yfinance gdown pyarrow

In [ ]:
import io, json, re
import numpy as np
import pandas as pd
import requests
import yfinance as yf

HEADERS={"User-Agent": "Mozilla/5.0"}
answers = {}
def fetch_html_tables(url):
    r=requests.get(url,headers=HEADERS,timeout=40)
    r.raise_for_status()
    return pd.read_html(io.StringIO(r.text))
def get_close(ticker, start="2025-01-01", end="2026-09-12"):
    d=yf.download(ticker,start=start,end=end,auto_adjust=False,progress=False,threads=False)
    if d.empty: return None
    c=d["Close"]
    c=c.iloc[:,0] if isinstance(c,pd.DataFrame) else c
    return c.dropna()
def numeric(x):
    try: return float(re.sub(r"[^0-9.\-]","",str(x)))
    except ValueError: return np.nan
print("pandas",pd.__version__,"yfinance",yf.__version__)

## Q1 — Withdrawn IPOs by company class
Follow ordered substring matching exactly: Technologies → Acquisition Corp/Corporation/Corp → Inc/Incorporated → Group → Ltd/Limited → Holdings/Holding → Other.

In [ ]:
tables=fetch_html_tables("https://www.iposcoop.com/ipos-recently-filed/")
for i,t in enumerate(tables): print(i,list(map(str,t.columns))[:12],len(t))
ipo=next(t.copy() for t in tables if "Expected To Trade" in map(str,t.columns))
withdrawn=ipo[ipo["Expected To Trade"].astype(str).str.strip().str.casefold().eq("withdrawn")].copy()
name_col = next(c for c in withdrawn if "company" in str(c).lower() and "type" not in str(c).lower())
rules=[("Technologies",["Technologies"]),("Acquisition Corp",["Acquisition Corp","Acquisition Corporation","Corp"]),("Inc.",["Inc","Incorporated"]),("Group",["Group"]),("Limited",["Ltd","Limited"]),("Holdings",["Holdings","Holding"])]
def classify(name):
    name=str(name)
    return next((group for group, patterns in rules if any(p in name for p in patterns)), "Other")
withdrawn["Company Type"]=withdrawn[name_col].map(classify)
withdrawn["price_low"]=withdrawn["Price Low"].map(numeric)
withdrawn["price_high"]=withdrawn["Price High"].map(numeric)
withdrawn["Avg_price"]=withdrawn[["price_low","price_high"]].mean(axis=1)
withdrawn["shares_m"]=withdrawn["Shares (millions)"].map(numeric)
vol_col=next(c for c in withdrawn if "Vol (millions)" in str(c))
withdrawn["fallback_value"]=withdrawn[vol_col].map(numeric)
withdrawn["Shares_offered_value"]=(withdrawn["shares_m"]*withdrawn["Avg_price"]).fillna(withdrawn["fallback_value"])
totals=withdrawn.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)
display(totals)
answers["q1_class"]=str(totals.index[0])
answers["q1_usd_millions"]=float(totals.iloc[0])
print("Withdrawn rows:",len(withdrawn),"Q1:",answers["q1_class"],answers["q1_usd_millions"])

## Q2 — IPO Sharpe on 11 September 2026
From 2025 pricings, use offers before 2025-09-01, exclude 0% return, then retrieve available Yahoo daily closes. Formula is copied *literally* from the assignment, including price-level rolling standard deviation (not conventional return volatility).

In [ ]:
tables=fetch_html_tables("https://www.iposcoop.com/2025-pricings/")
for i,t in enumerate(tables): print(i,list(map(str,t.columns))[:14],len(t))
ipos=next(t.copy() for t in tables if "Offer Date" in map(str,t.columns))
ipos["Offer Date"]=pd.to_datetime(ipos["Offer Date"],errors="coerce")
ipo_filtered=ipos[ipos["Offer Date"]<"2025-09-01"].copy()
return_col=next((c for c in ipo_filtered if "return" in str(c).lower()), None)
if return_col is None: raise KeyError("Inspect source: missing IPO return column")
ipo_filtered=ipo_filtered[ipo_filtered[return_col].map(numeric).ne(0)]
ticker_col=next((c for c in ipo_filtered if str(c).lower() in ("ticker","symbol")),None)
if ticker_col is None: raise KeyError("Inspect source: missing ticker/symbol column")
symbols=sorted(set(ipo_filtered[ticker_col].astype(str).str.strip())-{"","nan","-" })
print("Candidate tickers",len(symbols),"first examples",symbols[:10])
all_stocks=[]
for k,sym in enumerate(symbols,1):
    try:
        c=get_close(sym)
        if c is not None and len(c)>0:
            all_stocks.append(pd.DataFrame({"Date":pd.to_datetime(c.index).tz_localize(None).normalize(),"Ticker":sym,"Close":c.to_numpy()}))
    except Exception as exc:
        print("Skipping",sym,str(exc)[:80])
    if k%30==0: print("Fetched",k,"/",len(symbols))
assert all_stocks, "No IPO closing prices retrieved"
stocks_df=pd.concat(all_stocks,ignore_index=True).sort_values(["Ticker","Date"])
print("Yahoo tickers with data:",stocks_df["Ticker"].nunique())
stocks_df["growth_252d"]=stocks_df.groupby("Ticker")["Close"].transform(lambda s:s/s.shift(252))
stocks_df["volatility"]=stocks_df.groupby("Ticker")["Close"].transform(lambda s:s.rolling(30).std()*np.sqrt(252))
stocks_df["Sharpe"]=(stocks_df["growth_252d"]-0.05)/stocks_df["volatility"]
on_date=stocks_df[stocks_df["Date"].eq(pd.Timestamp("2026-09-11"))]
display(on_date[["growth_252d","volatility","Sharpe"]].describe())
answers["q2_median_sharpe"]=float(on_date["Sharpe"].median())
print("Q2:",answers["q2_median_sharpe"])

## Q3 — IPO holding duration, 1–12 months
Use each ticker's **first available trading close** as entry and its corresponding horizon forward return. Each month = 21 trading days.

In [ ]:
s=stocks_df.copy().sort_values(["Ticker","Date"])
for m in range(1,13):
    s[f"future_growth_{m}_m"]=s.groupby("Ticker")["Close"].shift(-21*m)/s["Close"]
ipo_first=s.drop_duplicates("Ticker",keep="first").copy()
hcols=[f"future_growth_{m}_m" for m in range(1,13)]
medians=ipo_first[hcols].median()
print("Median forward growth (ratio):")
display(medians)
answers["q3_optimal_month"]=int(medians.idxmax().split("_")[-2])
answers["q3_max_median_growth"]=float(medians.max())
print("Q3:",answers["q3_optimal_month"],answers["q3_max_median_growth"])

## Q4 — RSI-based strategy
Use the **official supplied Parquet**, not the IPO table. The dataset is downloaded only when this cell is run. RSI < 30, dates from 2000-01-01 up to but excluding 2025-06-01, $1,000 per trade.

In [ ]:
import gdown
file_id="1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}","data.parquet",quiet=False)
ind=pd.read_parquet("data.parquet",engine="pyarrow")
print("Columns:",ind.columns.tolist())
date_col=next(c for c in ind if str(c).lower() in ("date","datetime"))
rsi_col=next((c for c in ind if str(c).lower()=="rsi"),None)
if rsi_col is None: raise KeyError("Check RSI column name in supplied parquet")
target="growth_future_30d"
assert target in ind.columns, "Missing required growth_future_30d"
dates=pd.to_datetime(ind[date_col],errors="coerce")
chosen=ind.loc[(dates>="2000-01-01")&(dates<"2025-06-01")&(pd.to_numeric(ind[rsi_col],errors="coerce")<30)].copy()
chosen[target]=pd.to_numeric(chosen[target],errors="coerce")
chosen=chosen.dropna(subset=[target])
net_usd=1000*(chosen[target]-1).sum()
answers["q4_net_usd"]=float(net_usd)
answers["q4_net_usd_thousands"]=float(net_usd/1000)
print("Trades:",len(chosen),"Net (USD):",net_usd,"Net (thousands USD):",net_usd/1000)

## Q5 (Exploratory)
Improve IPO selection using sector-adjusted momentum, post-listing volatility, liquidity, listing valuation and macro conditions. Evaluate IPOs with **time-based walk-forward validation** and include delisted tickers, transaction costs and slippage to reduce survivorship and lookahead bias.

## Summary
This notebook runs no calculations when merely committed to GitHub. Values become usable only after running it and checking exact form options.

In [ ]:
print(json.dumps(answers,indent=2,default=str))